# 提示词模版(PromptTemplate)

## 1. 为什么推荐提示词模板？
在 LangChain 开发中，构造提示词既可以直接使用 Python 字符串拼接（如 f-string、format() 或+），也可以使用 LangChain 提供的 PromptTemplate 或 ChatPromptTemplate 。

举例一: 字符串拼接方式

In [3]:
import time
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 字符串拼接
topic = "Python"
difficulty = "初学者"
# 难以维护，容易出错
prompt_str = f"你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。"


load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)


response = model.invoke(prompt_str)
print(f"AI 回复：{response.content}...\n")

AI 回复：Python 是一种编程语言，可以用来告诉计算机完成任务。它的语法比较接近日常语言，因此常被初学者用来入门。

例如，下面这行代码会在屏幕上显示一条消息：

```python
print("你好，世界！")
```

几个常见概念：

- **变量**：给数据取个名字，比如 `name = "小明"`。
- **条件判断**：根据情况做不同的事，比如 `if age >= 18:`。
- **循环**：重复执行一段代码，比如 `for` 循环。
- **函数**：把一组操作打包起来，方便重复使用。

Python 代码通常从上往下执行。你可以把它想成给计算机写一份步骤清单：每一行告诉计算机下一步该做什么。...



优点✅：
* 简单直接，上手快
* 适合临时 demo无额外学习成本

缺点❌：
* 可读性差（变量多时混乱）
* 不易维护（修改容易出错）
* 无变量校验（容易漏/拼错）
* 难以支持复杂场景（多轮对话 / RAG / Few-shot）


举例2:提示词模版

In [2]:
from langchain_core.prompts import PromptTemplate

topic = "Python"
difficulty = "初学者"
template = PromptTemplate.from_template(
    "你是一个{difficulty}级别的编程导师。请用简单易懂的语言解释{topic}。"
)
# 使用模板生成提示词
prompt = template.format(difficulty=difficulty, topic=topic)
response = model.invoke(prompt)
print(f"AI 回复：{response.content}...\n")

AI 回复：Python 是一种编程语言，可以用来告诉电脑按步骤完成任务。它常用于自动化、数据分析、网站开发和人工智能。

比如这段代码：

```python
name = "小明"
print("你好，" + name)
```

它做了两件事：

- `name = "小明"`：把文字“小明”存进一个叫 `name` 的变量里。变量就像贴了标签的盒子。
- `print(...)`：让电脑把内容显示出来。

运行后会看到：

```text
你好，小明
```

Python 的语法通常比较接近日常语言。比如判断一个数字是否大于 10：

```python
number = 12

if number > 10:
    print("这个数字大于 10")
```

这里 `if` 表示“如果”。注意，`print` 前面的缩进表示这行代码属于这个条件；在 Python 里，缩进很重要。

可以先记住三个概念：**变量**用来保存信息，**条件**用来做判断，**循环**用来重复做事。...



In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate([
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "{user_input}")
])

prompt = prompt_template.format(name="豆包AI", user_input="你好，请问能帮我做什么?")
print(prompt)

System: 你是一个AI开发工程师. 你的名字是 豆包AI.
Human: 你好，请问能帮我做什么?


优点✅：
* 结构清晰（变量占位）
* 易维护、可复用
* 自动变量校验（更安全）
* 支持复杂场景（对话 / RAG / Agent）
* 可与 LangChain 生态无缝集成
* 便于调试与日志追踪

缺点❌：
* 有一定学习成本
* 初期写法略复杂
* 对极简单场景略“重”开发建议：小项目 / 临时用 → 字符串拼接
* 正式开发 / AI应用 → 提示词模板（必选）


## 2. 提示词机制演进

LangChain 1.0的架构变革中，核心的演进之一体现在 Prompt 机制上：一个结构化的、富含元数据的消息列表已经取代单一字符串，成为与模型交互的标准数据格式。

1. 旧时代：LLM + PromptTemplate（输入与输出均为字符串）
* 模型接口：对应于 LangChain 中的LLM类，主要面向早期的 文本补全模型 。
* 工作方式：模型接受一个 单一的字符串 作为输人，基于此预测并生成后续的文
本内容（文本补全）。
*  Prompt 工具：核心工具是 PromptTemplate 。它的职责是接收一组变量，并通过模板渲染，最终输出一个完整的字符串。
* 局限性：当我们需要用这种方式模拟多轮聊天时，开发者必须在字符串中手动拼接和伪造对话角色，例如：

这种方式不仅导致Prompt 的结构混乱、难以维护，也极易让模型混淆对话的边界与上下文，影响生成
质量。

2. 新时代：ChatModel+ChatPromptTemplate（输入与输出均为消息列表）

① 模型接口：对应 LangChain 1.0 的主流接口 ChatModel。

② 工作方式：现代聊天模型 API 已原生 支持角色概念 。它们不再接受单一字符串，
而是要求输入**一个结构化的消息列表**。为构建复杂、可靠的多轮对话智能体系统奠定了坚实的基础。

③ Prompt 工具： ChatPromptTemplate 因此成为LangChain 1.0 中最核心的 Prompt工具。它的职责是接收变量，并输出一个 List「BaseMessage］（消息列表），该列表可直接传递给聊天模型。

因此，用于生成消息列表的 ChatPromptTemplate，也自然取代了生成字符串的 PromptTemplate，成为构建现代LangChain 应用的首选工具。

## 3. ChatPromptTemplate的使用

在LangChain 1.0中，ChatPromptTemplate 是用于生成消息列表的核心组件。ChatPromptTemplate是创建 聊天消息列表 的提示模板。它比普通 PromptTemplate 更适合处理多角色、多轮次的对话场景。支持 System / Human / AI 等不同角色的消息模板。

### 3.1 两种实例化方式

1. 方式1(推荐)：调用from_messages() -> 推荐

该方法允许传入一个由元组（Tuple）构成的列表，列表中的每一个元组都代表一条具有特定角色的消息。


In [5]:
# 导入相关依赖
from langchain_core.prompts import ChatPromptTemplate
# 定义聊天提示词模版
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个有帮助的AI机器人，你的名字是{name}。"),
        ("human", "你好，最近怎么样？"),
        ("ai", "我很好，谢谢！"),
        ("human", "{user_input}"),
    ]
)
# 格式化聊天提示词模版中的变量
prompt = chat_template.invoke({"name":"小明", "user_input":"你叫什么名字？"})
# 打印格式化后的聊天提示词模版内容
print(prompt)

messages=[SystemMessage(content='你是一个有帮助的AI机器人，你的名字是小明。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='我很好，谢谢！', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你叫什么名字？', additional_kwargs={}, response_metadata={})]


2. 方式2：使用实例初始化方法

In [6]:
from langchain_core.prompts import ChatPromptTemplate
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])
#调用invoke()方法，返回ChatPromptValue
prompt = prompt_template.invoke({"name":"小谷AI", "user_input":"你能帮我做什么?"})
print(prompt)

messages=[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]


### 3.2 模板调用的3种方式

对比: invoke() 、 format() 、 format_messages()

方式1：使用 invoke()

传入: 字典

返回: ChatPromptValue。

在 LangChain 中，ChatPromptTemplate 的 invoke() 方法用于传入变量并渲染填充聊天提示词模板，最终生成可以直接传给大语言模型（LLM）的聊天消息对象。

In [7]:
from langchain_core.prompts import ChatPromptTemplate
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])
prompt = prompt_template.invoke({"name":"小谷AI", "user_input":"你能帮我做什么?"})
print(type(prompt))
print(prompt)
print(len(prompt.messages))

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]
4


方式2：使用format()

输入参数: 关键变量值

返回值: 字符串

In [8]:
from langchain_core.prompts import ChatPromptTemplate
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
# 字符串 role + 字符串 content
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])

#方式1：调用format()方法，返回字符串
prompt = prompt_template.format(name="小谷AI", user_input="你能帮我做什么?")
print(type(prompt))
print(prompt)

<class 'str'>
System: 你是一个AI开发工程师. 你的名字是 小谷AI.
Human: 你能开发哪些AI应用?
AI: 我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.
Human: 你能帮我做什么?


3. 方式三：使用format_messages()

输入参数: 关键变量值

返回值: 消息列表


In [9]:
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate([
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])
#调用format_messages()方法，返回消息列表
prompt = prompt_template.format_messages(name="小谷AI", user_input="你能帮我做什么?")
print(type(prompt))
print(prompt)

<class 'list'>
[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]


## 4. 大模型的调用

In [10]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
import os
from langchain_openai import ChatOpenAI
######1、提供大模型#########
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")


OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

######2、提供提示词#########
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个数学家，你可以计算任何算式"),
    ("human", "{text}"),
])

# 输入提示
prompt_value = chat_prompt.invoke({
    "text":"我今年18岁，我的舅舅今年38岁，我的爷爷今年72岁，我和舅舅一共多少岁了？"
})

######3、结合提示词，调用大模型#########
# 得到模型的输出
output = model.invoke(prompt_value)
# 打印输出内容
print(output.content)

18 + 38 = **56岁**。


## 4. 更丰富的初始化参数类型

前面讲了ChatPromptTemplate的两种创建方式。我们看到不管使用实例初始化方法，还是使用from_messages()，参数类型都是 列表类型 。列表中的元素可以是多种类型，前面我们主要测试了元组类型。

```python
    def __init__(
        self,
        messages: Sequence[MessageLikeRepresentation],
        *,
        template_format: PromptTemplateFormat = "f-string",
        **kwargs: Any,
    ) -> None:

    # 方式一源码
def from_messages(
        cls,
        messages: Sequence[MessageLikeRepresentation],
        template_format: PromptTemplateFormat = "f-string",
    ) -> ChatPromptTemplate:
```

在上面的 `MessageLikeRepresentation` 中，我们可以看到:
```python
MessageLikeRepresentation = BaseMessage | list[str] | tuple[str, str | list[str | dict[str, Any]]] | str | dict[str, Any]
```

类型一 : str列表类型

列表参数格式是str类型（不推荐），因为默认角色都是human

In [11]:
#1.导入相关依赖
from langchain_core.prompts import ChatPromptTemplate
# 2.定义聊天提示词模版
chat_template = ChatPromptTemplate.from_messages([
        "Hello, {name}!" # 等价于 ("human", "Hello, {name}!")
])
# 3. 使用invoke执行
messages = chat_template.invoke({"name":"小谷AI"})
# 4.打印格式化后的聊天提示词模版内容
print(messages)

messages=[HumanMessage(content='Hello, 小谷AI!', additional_kwargs={}, response_metadata={})]


类型二: 元组列表类型

In [12]:
# 示例: 元组形式的消息
prompt = ChatPromptTemplate.from_messages([
    ("system", "你的名字是{role}."),
    ("human", "很高兴认识你"),
])
print(prompt.invoke({"role":"小智"}))

messages=[SystemMessage(content='你的名字是小智.', additional_kwargs={}, response_metadata={}), HumanMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={})]


类型三: 字典列表类型

In [13]:
# 示例: 字典形式的消息
prompt = ChatPromptTemplate.from_messages([
    {"role": "system", "content": "你的名字是{role}."},
    {"role": "human", "content":"很高兴认识你"},
])
print(prompt.invoke({"role":"小智"}))

messages=[SystemMessage(content='你的名字是小智.', additional_kwargs={}, response_metadata={}), HumanMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={})]


类型四: Message列表类型

In [15]:
from langchain_core.messages import SystemMessage,HumanMessage
chat_prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage(content="我是一个贴心的智能助手"),
    HumanMessage(content="我的问题是:人工智能英文怎么说？")
])
messages = chat_prompt_template.invoke({})# invoke必须有一个dict参数
print(messages)
print(type(messages))

messages=[SystemMessage(content='我是一个贴心的智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:人工智能英文怎么说？', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


注意：在XxxMessage中不能有占位符。即：

In [16]:
from langchain_core.messages import SystemMessage,HumanMessage
chat_prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage(content="我是一个贴心的智能助手"),
    HumanMessage(content="我的问题是:{word}英文怎么说？")
])
messages = chat_prompt_template.invoke({"word":"人工智能"})
print(messages)
print(type(messages))

messages=[SystemMessage(content='我是一个贴心的智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='我的问题是:{word}英文怎么说？', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


类型五：MessagePromptTemplate列表类型

LangChain提供不同类型的MessagePromptTemplate。最常用的是SystemMessagePromptTemplate 、 HumanMessagePromptTemplate 和AIMessagePromptTemplate ，分别创建系统消息、人工消息和AI消息。

In [17]:
# 导入聊天消息类模板
from langchain_core.prompts import ChatPromptTemplate,HumanMessagePromptTemplate, SystemMessagePromptTemplate
# 创建消息模板
system_message_prompt = SystemMessagePromptTemplate.from_template("你是一个{role}")
human_message_prompt = HumanMessagePromptTemplate.from_template("给我解释{concept}，用浅显易懂的语言")
# 组合成聊天提示模板
chat_prompt = ChatPromptTemplate.from_messages([
    system_message_prompt,
    human_message_prompt
])
# 格式化提示
formatted_messages = chat_prompt.invoke({"role":"物理学家","concept":"相对论"})
print(formatted_messages)

messages=[SystemMessage(content='你是一个物理学家', additional_kwargs={}, response_metadata={}), HumanMessage(content='给我解释相对论，用浅显易懂的语言', additional_kwargs={}, response_metadata={})]


类型6：BaseChatPromptTemplate列表类型

使用 BaseChatPromptTemplate，可以理解为ChatPromptTemplate里嵌套了ChatPromptTemplate。

In [18]:
from langchain_core.prompts import ChatPromptTemplate
# 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt_template1 = ChatPromptTemplate.from_messages([
    ("system", "我是一个人工智能助手，我的名字叫{name}")
])
nested_prompt_template2 = ChatPromptTemplate.from_messages([
    ("human", "很高兴认识你,我的问题是{question}")
])
prompt_template = ChatPromptTemplate.from_messages([
    nested_prompt_template1,nested_prompt_template2
])

prompt_template.invoke({"name":"小智","question":"你为什么这么帅？"})


ChatPromptValue(messages=[SystemMessage(content='我是一个人工智能助手，我的名字叫小智', additional_kwargs={}, response_metadata={}), HumanMessage(content='很高兴认识你,我的问题是你为什么这么帅？', additional_kwargs={}, response_metadata={})])

In [19]:
from langchain_core.prompts import ChatPromptTemplate
# 使用 BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt_template1 = ChatPromptTemplate.from_messages([("system", "我是一个人工智能助手")])
nested_prompt_template2 = ChatPromptTemplate.from_messages([("human", "很高兴认识你")])
prompt_template = ChatPromptTemplate.from_messages([
nested_prompt_template1,nested_prompt_template2
])
prompt_template.invoke({})

ChatPromptValue(messages=[SystemMessage(content='我是一个人工智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='很高兴认识你', additional_kwargs={}, response_metadata={})])

综合使用

In [20]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)
from langchain_core.messages import SystemMessage, HumanMessage
import os

# 使用BaseMessage(已实例化的消息)
system_msg = SystemMessage("你是一个专业的AI工程师")
human_msg = HumanMessage("你好!")

# 示例二: 使用BaseMessagePromptTemplate(已实例化的消息)和ChatPromptTemplate
system_prompt = SystemMessagePromptTemplate.from_template("你是一个{role}")
human_prompt = HumanMessagePromptTemplate.from_template("{question}")

# 示例三: 使用BaseChatPromptTemplate（嵌套的 ChatPromptTemplate）
nested_prompt = ChatPromptTemplate.from_messages([
    ("system", "嵌套提示词"),
])

prompt = ChatPromptTemplate.from_messages([
    system_msg,
    human_msg,
    system_prompt,
    human_prompt,
    nested_prompt,
])

prompt.invoke({"role":"生物信息学专家","question":"介绍一下遗传分析"})


ChatPromptValue(messages=[SystemMessage(content='你是一个专业的AI工程师', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好!', additional_kwargs={}, response_metadata={}), SystemMessage(content='你是一个生物信息学专家', additional_kwargs={}, response_metadata={}), HumanMessage(content='介绍一下遗传分析', additional_kwargs={}, response_metadata={}), SystemMessage(content='嵌套提示词', additional_kwargs={}, response_metadata={})])